In [21]:
import os
import json
import numpy as np
import pandas as pd

In [22]:
DATA_FOLDER = "./Data/SOM"   # ajusta si cambia
START_YEAR = 1980
END_YEAR = 2024

FEATURES = [
    ("weights", "WoS Categories"),
    ("weights", "Document Types"),
    ("weights", "Documents,"),
    ("scores",  "Ave. citations"),
    ("scores",  "Ave. authorships"),
    ("scores",  "Ave. references"),
    ("scores",  "Percent of documents"),
    ("scores",  "Percent of documents Int. Coll.")
]

In [23]:

def is_ego_node(node):
    return node["id"].startswith("MCT") or node.get("scores", {}).get("Percent of documents", 0) == 100.0

def parse_microradial(raw_json):
    items = raw_json["network"]["items"]
    links = raw_json["network"]["links"]

    # Encontrar ego
    ego_indices = [i for i, n in enumerate(items) if is_ego_node(n)]
    if len(ego_indices) != 1:
        raise ValueError(f"Se esperaría 1 ego, pero encontré {len(ego_indices)}")

    ego_idx = ego_indices[0]
    ego_id = items[ego_idx]["id"]

    # máscara de alters
    alter_mask = np.ones(len(items), dtype=bool)
    alter_mask[ego_idx] = False

    # matriz X de alters (N x 8) N es el numero de alters de la red
    X = []
    clusters = []
    ids = []
    for i, node in enumerate(items):
        if not alter_mask[i]:
            continue
        row = []
        for scope, key in FEATURES:
            row.append(node.get(scope, {}).get(key, 0))
        X.append(row)
        clusters.append(node.get("cluster", -1))
        ids.append(node["id"])

    X = np.asarray(X, dtype=float)
    clusters = np.asarray(clusters)
    ids = np.asarray(ids)

    # strengths hacia el ego por nodo alter
    # source_id = alter, target_id = ego
    strength_by_alter = {}
    for link in links:
        if link["target_id"] == ego_id:
            strength_by_alter[link["source_id"]] = link.get("strength", 0)

    strengths = np.array([strength_by_alter.get(aid, 0) for aid in ids], dtype=float)

    return {
        "ego_id": ego_id,
        "X_alters": X,
        "clusters_alters": clusters,
        "alter_ids": ids,
        "strengths_to_ego": strengths,
        "n_items_total": len(items),
        "n_links_total": len(links)
    }


In [24]:
def validate_snapshot(parsed, year):
    """
    Lanza ValueError si algo crítico falla.
    Regresa un dict de warnings si hay suceden cosas inesperadas pero tolerables.
    """
    warnings = {}

    X = parsed["X_alters"]
    if X.ndim != 2:
        raise ValueError(f"[{year}] X_alters no es 2D (shape={X.shape}).")
    if X.shape[1] != len(FEATURES):
        raise ValueError(f"[{year}] X_alters tiene {X.shape[1]} features, esperaba {len(FEATURES)}.")
    if X.shape[0] == 0:
        raise ValueError(f"[{year}] No hay alters (X_alters vacío).")

  
    nan_count = int(np.isnan(X).sum())
    inf_count = int(np.isinf(X).sum())
    if nan_count > 0 or inf_count > 0:
        warnings["nan_inf"] = {"nan": nan_count, "inf": inf_count}

    strengths = parsed["strengths_to_ego"]
    if strengths.shape[0] != X.shape[0]:
        raise ValueError(f"[{year}] strengths_to_ego no coincide con #alters.")

    clusters = parsed["clusters_alters"]
    if clusters.shape[0] != X.shape[0]:
        raise ValueError(f"[{year}] clusters_alters no coincide con #alters.")

    return warnings

In [25]:
def build_yearly_index(data_folder, start_year, end_year):
    """
    Devuelve:
      - snapshots: dict[year] -> parsed snapshot
      - df_index: DataFrame con resumen anual
      - df_issues: DataFrame con errores/warnings por año
    """
    snapshots = {}
    index_rows = []
    issue_rows = []

    for year in range(start_year, end_year + 1):
        file_path = os.path.join(data_folder, f"network_{year}.json")

        if not os.path.exists(file_path):
            issue_rows.append({
                "year": year,
                "type": "missing_file",
                "detail": f"Archivo no encontrado: {file_path}"
            })
            continue

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                raw_json = json.load(f)

            parsed = parse_microradial(raw_json)
            warnings = validate_snapshot(parsed, year)

            snapshots[year] = parsed

            # Resumen anual
            X = parsed["X_alters"]
            clusters = parsed["clusters_alters"]
            strengths = parsed["strengths_to_ego"]

            index_rows.append({
                "year": year,
                "ego_id": parsed["ego_id"],
                "n_items_total": parsed["n_items_total"],
                "n_links_total": parsed["n_links_total"],
                "n_alters": X.shape[0],
                "n_features": X.shape[1],
                "n_clusters_unique": int(len(set(clusters.tolist()))),
                "sum_strength_to_ego": float(np.sum(strengths)),
                "mean_strength_to_ego": float(np.mean(strengths)),
                "max_strength_to_ego": float(np.max(strengths)),
            })

            # Guardar warnings si existen
            if warnings:
                issue_rows.append({
                    "year": year,
                    "type": "warning",
                    "detail": json.dumps(warnings, ensure_ascii=False)
                })

            print(f"[OK] {year} | alters={X.shape[0]} | ego={parsed['ego_id']}")

        except Exception as e:
            issue_rows.append({
                "year": year,
                "type": "error",
                "detail": str(e)
            })
            print(f"[ERROR] {year}: {e}")

    df_index = pd.DataFrame(index_rows).sort_values("year").reset_index(drop=True)
    df_issues = pd.DataFrame(issue_rows).sort_values(["year", "type"]).reset_index(drop=True)
    return snapshots, df_index, df_issues

snapshots, df_index, df_issues = build_yearly_index(
    data_folder=DATA_FOLDER,
    start_year=START_YEAR,
    end_year=END_YEAR
)

print("\n--- Resumen (df_index) ---")
print(df_index.head())

print("\n--- Issues/Warns (df_issues) ---")
print(df_issues.head(20))

print(f"\nAños procesados correctamente: {len(snapshots)} / {END_YEAR - START_YEAR + 1}")

[OK] 1980 | alters=26 | ego=MCT.4.61.1335
[OK] 1981 | alters=63 | ego=MCT.4.61.1335
[OK] 1982 | alters=58 | ego=MCT.4.61.1335
[OK] 1983 | alters=35 | ego=MCT.4.61.1335
[OK] 1984 | alters=45 | ego=MCT.4.61.1335
[OK] 1985 | alters=33 | ego=MCT.4.61.1335
[OK] 1986 | alters=18 | ego=MCT.4.61.1335
[OK] 1987 | alters=286 | ego=MCT.4.61.1335
[OK] 1988 | alters=128 | ego=MCT.4.61.1335
[OK] 1989 | alters=61 | ego=MCT.4.61.1335
[ERROR] 1990: Se esperaría 1 ego, pero encontré 2
[OK] 1991 | alters=244 | ego=MCT.4.61.1335
[OK] 1992 | alters=224 | ego=MCT.4.61.1335
[OK] 1993 | alters=103 | ego=MCT.4.61.1335
[OK] 1994 | alters=63 | ego=MCT.4.61.1335
[OK] 1995 | alters=167 | ego=MCT.4.61.1335
[OK] 1996 | alters=595 | ego=MCT.4.61.1335
[OK] 1997 | alters=150 | ego=MCT.4.61.1335
[OK] 1998 | alters=128 | ego=MCT.4.61.1335
[OK] 1999 | alters=93 | ego=MCT.4.61.1335
[OK] 2000 | alters=155 | ego=MCT.4.61.1335
[OK] 2001 | alters=127 | ego=MCT.4.61.1335
[OK] 2002 | alters=107 | ego=MCT.4.61.1335
[OK] 2003 | al

In [26]:
print("\n--- Resumen (df_index) ---")
print(df_index)


--- Resumen (df_index) ---
    year         ego_id  n_items_total  n_links_total  n_alters  n_features  \
0   1980  MCT.4.61.1335             27             26        26           8   
1   1981  MCT.4.61.1335             64             63        63           8   
2   1982  MCT.4.61.1335             59             58        58           8   
3   1983  MCT.4.61.1335             36             35        35           8   
4   1984  MCT.4.61.1335             46             45        45           8   
5   1985  MCT.4.61.1335             34             33        33           8   
6   1986  MCT.4.61.1335             19             18        18           8   
7   1987  MCT.4.61.1335            287            286       286           8   
8   1988  MCT.4.61.1335            129            128       128           8   
9   1989  MCT.4.61.1335             62             61        61           8   
10  1991  MCT.4.61.1335            245            244       244           8   
11  1992  MCT.4.61.1335 

In [27]:
def safe_probs(masses: np.ndarray):
    """
    Convierte masas >=0 en distribución p_i.
    Regresa None si no se puede (suma 0 o NaN).
    """
    masses = np.asarray(masses, dtype=float)
    masses = np.where(np.isfinite(masses), masses, 0.0)
    masses = np.where(masses < 0, 0.0, masses)

    total = masses.sum()
    if total <= 0:
        return None
    return masses / total

In [28]:
def hill_numbers(p: np.ndarray):
    """
    Calcula Hill numbers para q=0,1,2.
    p debe ser distribución (suma 1).
    Devuelve (D0, D1, D2).
    """
    if p is None:
        return (np.nan, np.nan, np.nan)

    p = np.asarray(p, dtype=float)
    p = p[p > 0]  # evitar log(0), pues no está definido
    if p.size == 0:
        return (np.nan, np.nan, np.nan)

    # q=0: riqueza (número de categorías con masa >0)
    D0 = float(p.size)

    # q=1: exp(Shannon)
    H = -float(np.sum(p * np.log(p)))
    D1 = float(np.exp(H))

    # q=2: inverse Simpson
    D2 = float(1.0 / np.sum(p ** 2))

    return (D0, D1, D2)

In [29]:
def group_masses_by_cluster(clusters: np.ndarray, masses: np.ndarray):
    """
    Suma masas por cluster.
    Regresa: clusters_unicos, masas_por_cluster
    """
    clusters = np.asarray(clusters)
    masses = np.asarray(masses, dtype=float)

    # limpiar masas
    masses = np.where(np.isfinite(masses), masses, 0.0)
    masses = np.where(masses < 0, 0.0, masses)

    df = pd.DataFrame({"cluster": clusters, "mass": masses})
    grp = df.groupby("cluster", as_index=False)["mass"].sum().sort_values("cluster") #Agrupa todos los que pertenecen al mismo cluster, y suma las masas de los integrantes, al final los odena
    return grp["cluster"].to_numpy(), grp["mass"].to_numpy()


In [30]:
def meso_stats_from_X(X: np.ndarray):
    """
    Calcula centro y dispersión de una nube de puntos (alters x features).
    Regresa dict con métricas globales + arrays de mean/var por feature.
    """
    X = np.asarray(X, dtype=float)
    # limpiar NaN/inf
    X = np.where(np.isfinite(X), X, np.nan)

    # mean por feature (ignora NaN)
    mu = np.nanmean(X, axis=0)

    # var por feature
    var = np.nanvar(X, axis=0)

    # cov y trace (si N>=2)
    if X.shape[0] >= 2:
        # para cov: reemplazar NaN por mean de columna para no romper el calculo, es una estrategia clasicca de estadistica
        X_filled = np.where(np.isnan(X), mu, X)
        cov = np.cov(X_filled, rowvar=False)
        trace_cov = float(np.trace(cov))
    else:
        trace_cov = np.nan

    # distancia promedio al centro
    X_filled = np.where(np.isnan(X), mu, X)
    dists = np.linalg.norm(X_filled - mu, axis=1)
    mean_dist = float(np.mean(dists))
    median_dist = float(np.median(dists))

    return {
        "mu": mu,
        "var": var,
        "trace_cov": trace_cov,
        "mean_dist_to_mu": mean_dist,
        "median_dist_to_mu": median_dist,
    }


In [31]:

def run_stage2(snapshots: dict, feature_names: list, documents_col_name="Documents,"):
    """
    Construye un DataFrame con métricas mesotópicas por año:
    - mean_* por feature
    - var_* por feature
    - trace_cov, mean_dist_to_mu, median_dist_to_mu
    - Hill numbers por clusters con masa Documents y con masa strength

    documents_col_name lo dejo como "Documents,"
    """
    # ubicar índice de la columna Documents,
    # feature_names debe corresponder a FEATURES en el mismo orden
    if documents_col_name not in feature_names:
        print(f"[AVISO] '{documents_col_name}' no está en feature_names. Hill por Documents saldrá NaN.")
        documents_idx = None
    else:
        documents_idx = feature_names.index(documents_col_name)

    rows = []

    for year in sorted(snapshots.keys()):
        snap = snapshots[year]
        X = snap["X_alters"]
        clusters = snap["clusters_alters"]
        strengths = snap["strengths_to_ego"]

        # --- A/B: centro y dispersión
        stats = meso_stats_from_X(X)
        mu = stats["mu"]
        var = stats["var"]

        row = {
            "year": year,
            "ego_id": snap["ego_id"],
            "n_alters": int(X.shape[0]),
            "trace_cov": stats["trace_cov"],
            "mean_dist_to_mu": stats["mean_dist_to_mu"],
            "median_dist_to_mu": stats["median_dist_to_mu"],
        }

        # mean_* y var_* por feature (interpretables)
        for j, fname in enumerate(feature_names):
            row[f"mean_{fname}"] = float(mu[j]) if np.isfinite(mu[j]) else np.nan
            row[f"var_{fname}"] = float(var[j]) if np.isfinite(var[j]) else np.nan

        # --- C: Hill numbers por clusters usando masa "Documents,"
        if documents_idx is not None:
            docs = X[:, documents_idx]
            _, mass_by_cluster_docs = group_masses_by_cluster(clusters, docs)
            p_docs = safe_probs(mass_by_cluster_docs)
            D0_docs, D1_docs, D2_docs = hill_numbers(p_docs)
        else:
            D0_docs = D1_docs = D2_docs = np.nan

        row["hill_docs_q0"] = D0_docs
        row["hill_docs_q1"] = D1_docs
        row["hill_docs_q2"] = D2_docs

        # Hill por clusters usando masa "strength_to_ego"
        _, mass_by_cluster_str = group_masses_by_cluster(clusters, strengths)
        p_str = safe_probs(mass_by_cluster_str)
        D0_str, D1_str, D2_str = hill_numbers(p_str)

        row["hill_strength_q0"] = D0_str
        row["hill_strength_q1"] = D1_str
        row["hill_strength_q2"] = D2_str

        # Controles rápidos de dominancia
        # participación del cluster más grande por docs/strength
        if documents_idx is not None and p_docs is not None:
            row["top1_share_docs"] = float(np.max(p_docs))
        else:
            row["top1_share_docs"] = np.nan

        if p_str is not None:
            row["top1_share_strength"] = float(np.max(p_str))
        else:
            row["top1_share_strength"] = np.nan

        rows.append(row)

    df_stage2 = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    return df_stage2

In [32]:

# Construye el "feature_names" en el mismo orden que FEATURES
feature_names = [k for (_, k) in FEATURES]  

df_stage2 = run_stage2(
    snapshots=snapshots,
    feature_names=feature_names,
    documents_col_name="Documents,"
)

print(df_stage2)
print("\nColumnas disponibles:", len(df_stage2.columns))

    year         ego_id  n_alters      trace_cov  mean_dist_to_mu  \
0   1980  MCT.4.61.1335        26    4614.191661        58.901404   
1   1981  MCT.4.61.1335        63    5596.362150        58.173082   
2   1982  MCT.4.61.1335        58   33617.304839       132.810891   
3   1983  MCT.4.61.1335        35   15985.350265        87.826122   
4   1984  MCT.4.61.1335        45   27442.864802       102.555862   
5   1985  MCT.4.61.1335        33   21439.316198       109.359888   
6   1986  MCT.4.61.1335        18    6545.682430        64.997296   
7   1987  MCT.4.61.1335       286   18688.695794        83.696426   
8   1988  MCT.4.61.1335       128   35830.598134       105.678275   
9   1989  MCT.4.61.1335        61   74713.286145       123.678260   
10  1991  MCT.4.61.1335       244    8249.031398        60.693496   
11  1992  MCT.4.61.1335       224   15205.079764        66.915939   
12  1993  MCT.4.61.1335       103   16787.878728        89.424200   
13  1994  MCT.4.61.1335        63 

In [33]:
# Se guarda en la misma carpeta donde está tu script
df_stage2.to_csv("./analisis_mesotopico.csv", index=False)